In [4]:
"""
collect_data.py
===============
Шаг 1 ETL: Скачиваем сырые данные из yfinance и FRED.
Результат: raw_gold_data.csv
"""

import pandas as pd
import yfinance as yf
from fredapi import Fred
import datetime


FRED_API_KEY = "308c46222636bb6d8743d4b4902a42d1"
START_DATE   = "2000-01-01"
END_DATE     = datetime.datetime.now().strftime("%Y-%m-%d")
OUTPUT_FILE  = "raw_gold_data.csv"

# ── Тикеры Yahoo Finance ──────────────────────────────────────────────────────
#
# Распределение по таблицам MySQL:
#   daily_prices   → GC=F, SI=F, DX-Y.NYB, ^GSPC, ^VIX, EURUSD=X, USDCNY=X,
#                    RUB=X, PL=F, PA=F
#   etf_flows      → GLD, IAU
#   energy_mining  → CL=F, BZ=F, HG=F

YF_TICKERS = {
    # Драгоценные металлы
    "GC=F":     "Gold_Price",
    "SI=F":     "Silver_Price",
    "PL=F":     "Platinum_Price",
    "PA=F":     "Palladium_Price",
    # Индексы
    "DX-Y.NYB": "DXY_Index",
    "^GSPC":    "SP500",
    "^VIX":     "VIX_Index",
    # Валюты
    "EURUSD=X": "USD_EUR",
    "USDCNY=X": "USD_CNY",
    "RUB=X":    "USD_RUB",
    # ETF золота
    "GLD":      "GLD_Price",
    "IAU":      "IAU_Price",
    # Энергетика и металлы (для energy_mining)
    "CL=F":     "WTI_Oil",
    "BZ=F":     "Brent_Oil",
    "HG=F":     "Copper_Price",
}

# ── Серии FRED ────────────────────────────────────────────────────────────────
#
# Все идут в таблицу macro_indicators.
# PPI также используется при расчёте Mining_Cost_Index (в transform_data.py).

FRED_SERIES = {
    "FEDFUNDS":         "Fed_Rate",           # Ставка ФРС
    "DGS10":            "Treasury_10Y",       # Доходность 10-летних UST
    "DGS2":             "Treasury_2Y",        # Доходность 2-летних UST
    "CPIAUCSL":         "CPI_Index",          # Индекс потребительских цен
    "UNRATE":           "Unemployment",       # Безработица
    "M2SL":             "M2_Supply",          # Денежная масса M2
    "PPIACO":           "PPI",                # PPI (нужен для mining cost)
}


def download_yfinance() -> pd.DataFrame:
    print(f"  Загружаем {len(YF_TICKERS)} тикеров из Yahoo Finance...")
    raw = yf.download(
        list(YF_TICKERS.keys()),
        start=START_DATE,
        end=END_DATE,
        auto_adjust=True
    )

    # Берём только Close цены
    if isinstance(raw.columns, pd.MultiIndex):
        df = raw["Close"].copy()
    else:
        df = raw.copy()

    # ── Исправление: убираем \n из названий колонок ──
    df.columns = pd.Index([str(col).strip() for col in df.columns])

    df = df.rename(columns=YF_TICKERS)
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.index.name = "Date"

    # Brent имеет историю только с 2006 — заполняем пропуски через WTI
    if "Brent_Oil" in df.columns and "WTI_Oil" in df.columns:
        df["Brent_Oil"] = df["Brent_Oil"].fillna(df["WTI_Oil"])

    print(f"  yfinance: {len(df)} строк, {df.shape[1]} колонок")
    print(f"  Колонки: {list(df.columns)}")
    return df


yf_df   = download_yfinance()
yf_df

  Загружаем 15 тикеров из Yahoo Finance...


[*********************100%***********************]  15 of 15 completed


  yfinance: 6835 строк, 15 колонок
  Колонки: ['Brent_Oil', 'WTI_Oil', 'DXY_Index', 'USD_EUR', 'Gold_Price', 'GLD_Price', 'Copper_Price', 'IAU_Price', 'Palladium_Price', 'Platinum_Price', 'USD_RUB', 'Silver_Price', 'USD_CNY', 'SP500', 'VIX_Index']


,Brent_Oil,WTI_Oil,DXY_Index,USD_EUR,Gold_Price,GLD_Price,Copper_Price,IAU_Price,Palladium_Price,Platinum_Price,USD_RUB,Silver_Price,USD_CNY,SP500,VIX_Index
Date,,,,,,,,,,,,,,,
2000-01-03,NaN,NaN,100.220001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1455.219971,24.209999
2000-01-04,NaN,NaN,100.410004,NaN,NaN,NaN,NaN,NaN,441.899994,429.700012,NaN,NaN,NaN,1399.420044,27.010000
2000-01-05,NaN,NaN,100.379997,NaN,NaN,NaN,NaN,NaN,438.100006,419.899994,NaN,NaN,NaN,1402.109985,26.410000
2000-01-06,NaN,NaN,100.650002,NaN,NaN,NaN,NaN,NaN,435.299988,412.000000,NaN,NaN,NaN,1403.449951,25.730000
2000-01-07,NaN,NaN,100.800003,NaN,NaN,NaN,NaN,NaN,443.899994,414.000000,NaN,NaN,NaN,1441.469971,21.719999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-06,109.769997,112.410004,99.980003,1.151000,4656.799805,427.649994,5.5835,87.610001,1476.199951,1958.199951,80.405930,72.661003,6.8824,6611.830078,24.170000
2026-04-07,109.269997,112.949997,99.639999,1.154121,4657.100098,431.809998,5.5445,88.459999,1442.000000,1929.099976,80.184731,71.825996,6.8823,6616.850098,25.780001
2026-04-08,94.750000,94.410004,99.129997,1.168661,4749.500000,434.529999,5.7595,89.040001,1586.900024,2050.600098,78.421890,75.223999,6.8567,6782.810059,21.040001


In [5]:
def download_fred() -> pd.DataFrame:
    """Скачивает макроданные из FRED."""
    print(f"  Загружаем {len(FRED_SERIES)} серий из FRED...")
    fred = Fred(api_key=FRED_API_KEY)
    frames = {}

    for code, name in FRED_SERIES.items():
        try:
            series = fred.get_series(code, observation_start=START_DATE)
            frames[name] = series
            print(f"    ✓ {code} → {name} ({len(series)} наблюдений)")
        except Exception as e:
            print(f"    ✗ {code}: {e}")

    df = pd.DataFrame(frames)
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.index.name = "Date"
    return df

fred_df = download_fred()
fred_df

  Загружаем 7 серий из FRED...
    ✓ FEDFUNDS → Fed_Rate (315 наблюдений)
    ✓ DGS10 → Treasury_10Y (6854 наблюдений)
    ✓ DGS2 → Treasury_2Y (6854 наблюдений)
    ✓ CPIAUCSL → CPI_Index (315 наблюдений)
    ✓ UNRATE → Unemployment (315 наблюдений)
    ✓ M2SL → M2_Supply (314 наблюдений)
    ✓ PPIACO → PPI (314 наблюдений)


,Fed_Rate,Treasury_10Y,Treasury_2Y,CPI_Index,Unemployment,M2_Supply,PPI
Date,,,,,,,
2000-01-01,5.45,NaN,NaN,169.3,4.0,4667.6,128.3
2000-01-03,NaN,6.58,6.38,NaN,NaN,NaN,NaN
2000-01-04,NaN,6.49,6.30,NaN,NaN,NaN,NaN
2000-01-05,NaN,6.62,6.38,NaN,NaN,NaN,NaN
2000-01-06,NaN,6.57,6.35,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
2026-04-03,NaN,4.35,3.84,NaN,NaN,NaN,NaN
2026-04-06,NaN,4.34,3.84,NaN,NaN,NaN,NaN
2026-04-07,NaN,4.33,3.81,NaN,NaN,NaN,NaN


In [7]:
def collect_data():
    print(f"=== collect_data.py: сбор данных {START_DATE} → {END_DATE} ===\n")

    yf_df   = download_yfinance()
    fred_df = download_fred()

    # Объединяем по индексу (дата).
    # FRED данные публикуются реже (месяц/квартал) — пропуски заполняются позже
    # в transform_data.py через ffill, здесь оставляем NaN как есть.
    df = pd.concat([yf_df, fred_df], axis=1)
    df = df.sort_index()

    df.to_csv(OUTPUT_FILE)
    print(f"\n=== Готово ===")
    print(f"  Файл:    {OUTPUT_FILE}")
    print(f"  Строк:   {len(df)}")
    print(f"  Колонок: {df.shape[1]}")
    print(f"  Период:  {df.index.min().date()} → {df.index.max().date()}")

collect_data()    

=== collect_data.py: сбор данных 2000-01-01 → 2026-04-13 ===

  Загружаем 15 тикеров из Yahoo Finance...


[*********************100%***********************]  15 of 15 completed


  yfinance: 6835 строк, 15 колонок
  Колонки: ['Brent_Oil', 'WTI_Oil', 'DXY_Index', 'USD_EUR', 'Gold_Price', 'GLD_Price', 'Copper_Price', 'IAU_Price', 'Palladium_Price', 'Platinum_Price', 'USD_RUB', 'Silver_Price', 'USD_CNY', 'SP500', 'VIX_Index']
  Загружаем 7 серий из FRED...
    ✓ FEDFUNDS → Fed_Rate (315 наблюдений)
    ✓ DGS10 → Treasury_10Y (6854 наблюдений)
    ✓ DGS2 → Treasury_2Y (6854 наблюдений)
    ✓ CPIAUCSL → CPI_Index (315 наблюдений)
    ✓ UNRATE → Unemployment (315 наблюдений)
    ✓ M2SL → M2_Supply (314 наблюдений)
    ✓ PPIACO → PPI (314 наблюдений)

=== Готово ===
  Файл:    raw_gold_data.csv
  Строк:   6947
  Колонок: 22
  Период:  2000-01-01 → 2026-04-10


C:\Users\ASUS\AppData\Local\Temp\ipykernel_15684\1203082829.py:10: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = pd.concat([yf_df, fred_df], axis=1)
